## 05 - Database Setup

This notebook creates our SQLite database and loads all data into it.

### What is SQLite?
SQLite is a lightweight database that lives in a single file on your computer.
No server needed, no complicated setup — just a file called earnings_research.db.

### Why do we need a database?
Right now our data is scattered across 50+ CSV files and 14 transcript files.
A database puts everything in one place and lets us query it instantly.

### Tables we create
1. companies — our 50 S&P 500 companies with sectors
2. stock_prices — daily price data for all 50 companies 2018-2025
3. transcripts — all 14 earnings call transcripts with metadata

In [1]:
# Import libraries
import sqlite3
import pandas as pd
import os

# Create database connection
# This creates the file if it doesn't exist
conn = sqlite3.connect("../data/earnings_research.db")
cursor = conn.cursor()

print("Database created successfully!")
print("Location: data/earnings_research.db")

Database created successfully!
Location: data/earnings_research.db


In [2]:
# Create companies table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS companies (
        ticker TEXT PRIMARY KEY,
        sector TEXT
    )
""")

conn.commit()
print("Companies table created!")

Companies table created!


In [3]:
# Load company list from CSV into database
df_companies = pd.read_csv("../data/company_list.csv")

# Insert each company into the database
df_companies.to_sql("companies", conn, if_exists="replace", index=False)

# Verify it worked
result = pd.read_sql("SELECT * FROM companies LIMIT 10", conn)
print(result)
print(f"\nTotal companies in database: {len(pd.read_sql('SELECT * FROM companies', conn))}")

  ticker      sector
0   AAPL  Technology
1   MSFT  Technology
2  GOOGL  Technology
3   META  Technology
4   NVDA  Technology
5   INTC  Technology
6    CRM  Technology
7   ADBE  Technology
8    AMD  Technology
9   ORCL  Technology

Total companies in database: 50


In [4]:
# Create stock prices table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS stock_prices (
        ticker TEXT,
        date TEXT,
        open REAL,
        high REAL,
        low REAL,
        close REAL,
        volume INTEGER
    )
""")

conn.commit()
print("Stock prices table created!")

Stock prices table created!


In [5]:
# Load all 50 stock price CSV files into database
prices_folder = "../data/prices"
failed = []

for filename in os.listdir(prices_folder):
    if filename.endswith(".csv"):
        ticker = filename.split("_")[0]
        filepath = os.path.join(prices_folder, filename)
        
        try:
            df = pd.read_csv(filepath)
            df["ticker"] = ticker
            
            # Keep only columns we need
            df = df[["ticker", "Date", "Open", "High", "Low", "Close", "Volume"]]
            df.columns = ["ticker", "date", "open", "high", "low", "close", "volume"]
            
            df.to_sql("stock_prices", conn, if_exists="append", index=False)
            print(f"✓ {ticker} loaded")
        
        except Exception as e:
            failed.append(ticker)
            print(f"✗ {ticker} failed: {e}")

conn.commit()
print(f"\nDone! Failed: {failed}")

✓ AAPL loaded
✓ AAPL loaded
✓ ABT loaded
✓ ADBE loaded
✓ AMD loaded
✓ AMGN loaded
✓ AMZN loaded
✓ AXP loaded
✓ BAC loaded
✓ BLK loaded
✓ BMY loaded
✓ COP loaded
✓ COST loaded
✓ CRM loaded
✓ CVX loaded
✓ C loaded
✓ DHR loaded
✓ DVN loaded
✓ EOG loaded
✓ GOOGL loaded
✓ GS loaded
✓ HAL loaded
✓ HD loaded
✓ INTC loaded
✓ JNJ loaded
✓ JPM loaded
✓ KO loaded
✓ MCD loaded
✓ MDT loaded
✓ META loaded
✓ MPC loaded
✓ MRK loaded
✓ MSFT loaded
✓ MS loaded
✓ NKE loaded
✓ NVDA loaded
✓ ORCL loaded
✓ OXY loaded
✓ PFE loaded
✓ PG loaded
✓ PNC loaded
✓ PXD loaded
✓ SBUX loaded
✓ SLB loaded
✓ TGT loaded
✓ TMO loaded
✓ UNH loaded
✓ USB loaded
✓ VLO loaded
✓ WFC loaded
✓ WMT loaded
✓ XOM loaded

Done! Failed: []


In [6]:
# Verify stock prices data
result = pd.read_sql("""
    SELECT ticker, COUNT(*) as days_of_data 
    FROM stock_prices 
    GROUP BY ticker 
    ORDER BY ticker
""", conn)

print(result)
print(f"\nTotal rows in stock_prices table: {len(pd.read_sql('SELECT * FROM stock_prices', conn))}")

   ticker  days_of_data
0    AAPL          2073
1     ABT          2010
2    ADBE          2010
3     AMD          2010
4    AMGN          2010
5    AMZN          2010
6     AXP          2010
7     BAC          2010
8     BLK          2010
9     BMY          2010
10      C          2010
11    COP          2010
12   COST          2010
13    CRM          2010
14    CVX          2010
15    DHR          2010
16    DVN          2010
17    EOG          2010
18  GOOGL          2010
19     GS          2010
20    HAL          2010
21     HD          2010
22   INTC          2010
23    JNJ          2010
24    JPM          2010
25     KO          2010
26    MCD          2010
27    MDT          2010
28   META          2010
29    MPC          2010
30    MRK          2010
31     MS          2010
32   MSFT          2010
33    NKE          2010
34   NVDA          2010
35   ORCL          2010
36    OXY          2010
37    PFE          2010
38     PG          2010
39    PNC          2010
40   SBUX       

In [7]:
# Fix AAPL duplicate - delete and reload clean
cursor.execute("DELETE FROM stock_prices WHERE ticker = 'AAPL'")
conn.commit()

# Reload just the correct AAPL file
df = pd.read_csv("../data/prices/AAPL_2018_2025.csv")
df["ticker"] = "AAPL"
df = df[["ticker", "Date", "Open", "High", "Low", "Close", "Volume"]]
df.columns = ["ticker", "date", "open", "high", "low", "close", "volume"]
df.to_sql("stock_prices", conn, if_exists="append", index=False)
conn.commit()

# Also remove PXD since we replaced it with DVN
cursor.execute("DELETE FROM stock_prices WHERE ticker = 'PXD'")
conn.commit()

# Verify
result = pd.read_sql("""
    SELECT ticker, COUNT(*) as days_of_data 
    FROM stock_prices 
    WHERE ticker IN ('AAPL', 'PXD', 'DVN')
""", conn)
print(result)
print(f"\nTotal rows now: {len(pd.read_sql('SELECT * FROM stock_prices', conn))}")

  ticker  days_of_data
0    DVN          4020

Total rows now: 100500


In [8]:
# Check what's happening with AAPL and DVN
result = pd.read_sql("""
    SELECT ticker, COUNT(*) as days_of_data 
    FROM stock_prices 
    GROUP BY ticker 
    ORDER BY ticker
""", conn)

print(result)
print(f"\nTotal rows: {len(pd.read_sql('SELECT * FROM stock_prices', conn))}")

   ticker  days_of_data
0    AAPL          2010
1     ABT          2010
2    ADBE          2010
3     AMD          2010
4    AMGN          2010
5    AMZN          2010
6     AXP          2010
7     BAC          2010
8     BLK          2010
9     BMY          2010
10      C          2010
11    COP          2010
12   COST          2010
13    CRM          2010
14    CVX          2010
15    DHR          2010
16    DVN          2010
17    EOG          2010
18  GOOGL          2010
19     GS          2010
20    HAL          2010
21     HD          2010
22   INTC          2010
23    JNJ          2010
24    JPM          2010
25     KO          2010
26    MCD          2010
27    MDT          2010
28   META          2010
29    MPC          2010
30    MRK          2010
31     MS          2010
32   MSFT          2010
33    NKE          2010
34   NVDA          2010
35   ORCL          2010
36    OXY          2010
37    PFE          2010
38     PG          2010
39    PNC          2010
40   SBUX       

In [9]:
# Create transcripts table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS transcripts (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        ticker TEXT,
        quarter TEXT,
        date TEXT,
        period TEXT,
        raw_text TEXT
    )
""")

conn.commit()
print("Transcripts table created!")

Transcripts table created!


In [10]:
# Load all clean transcripts into database
transcripts_folder = "../data/transcripts"

# Map filenames to metadata
transcript_metadata = {
    "AAPL_Q1_2020": {"ticker": "AAPL", "quarter": "Q1_2020", "date": "2020-01-28", "period": "Pre-COVID"},
    "AAPL_Q2_2020": {"ticker": "AAPL", "quarter": "Q2_2020", "date": "2020-04-30", "period": "COVID Crash"},
    "AAPL_Q4_2022": {"ticker": "AAPL", "quarter": "Q4_2022", "date": "2022-10-27", "period": "Rate Hike Peak"},
    "AAPL_Q1_2025": {"ticker": "AAPL", "quarter": "Q1_2025", "date": "2025-01-30", "period": "Tariff Shock"},
    "AMZN_Q1_2020": {"ticker": "AMZN", "quarter": "Q1_2020", "date": "2020-04-30", "period": "Pre-COVID"},
    "AMZN_Q1_2025": {"ticker": "AMZN", "quarter": "Q1_2025", "date": "2025-05-01", "period": "Tariff Shock"},
    "JNJ_Q1_2020":  {"ticker": "JNJ",  "quarter": "Q1_2020", "date": "2020-04-14", "period": "Pre-COVID"},
    "JPM_Q1_2020":  {"ticker": "JPM",  "quarter": "Q1_2020", "date": "2020-04-14", "period": "Pre-COVID"},
    "JPM_Q2_2020":  {"ticker": "JPM",  "quarter": "Q2_2020", "date": "2020-07-14", "period": "COVID Crash"},
    "MSFT_Q1_2020": {"ticker": "MSFT", "quarter": "Q1_2020", "date": "2020-04-29", "period": "Pre-COVID"},
    "MSFT_Q4_2022": {"ticker": "MSFT", "quarter": "Q4_2022", "date": "2022-07-26", "period": "Rate Hike Peak"},
    "MSFT_Q2_2023": {"ticker": "MSFT", "quarter": "Q2_2023", "date": "2023-01-24", "period": "AI Boom"},
    "NVDA_Q2_2023": {"ticker": "NVDA", "quarter": "Q2_2023", "date": "2023-08-23", "period": "AI Boom"},
    "XOM_Q1_2020":  {"ticker": "XOM",  "quarter": "Q1_2020", "date": "2020-05-01", "period": "Pre-COVID"},
}

count = 0
for key, meta in transcript_metadata.items():
    clean_file = f"{transcripts_folder}/{key}_clean.txt"
    
    try:
        with open(clean_file, "r", encoding="utf-8") as f:
            text = f.read()
        
        cursor.execute("""
            INSERT INTO transcripts (ticker, quarter, date, period, raw_text)
            VALUES (?, ?, ?, ?, ?)
        """, (meta["ticker"], meta["quarter"], meta["date"], meta["period"], text))
        
        count += 1
        print(f"✓ {key} loaded")
    
    except Exception as e:
        print(f"✗ {key} failed: {e}")

conn.commit()
print(f"\nTotal transcripts in database: {count}")

✓ AAPL_Q1_2020 loaded
✓ AAPL_Q2_2020 loaded
✓ AAPL_Q4_2022 loaded
✓ AAPL_Q1_2025 loaded
✓ AMZN_Q1_2020 loaded
✓ AMZN_Q1_2025 loaded
✓ JNJ_Q1_2020 loaded
✓ JPM_Q1_2020 loaded
✓ JPM_Q2_2020 loaded
✓ MSFT_Q1_2020 loaded
✓ MSFT_Q4_2022 loaded
✓ MSFT_Q2_2023 loaded
✓ NVDA_Q2_2023 loaded
✓ XOM_Q1_2020 loaded

Total transcripts in database: 14


In [12]:
# Let's query our database and explore it

#Query 1 - How many transcripts per market period?
print("=== Transcripts per market period ===")
result = pd. read_sql("""
    SELECT period, COUNT(*) as count
    FROM transcripts
    GROUP BY period
    ORDER BY date
""", conn)
print(result)

# Query 2 - Show all transcripts
print("\n=== All Transcripts ===")
result = pd.read_sql("""
     SELECT ticker, quarter, date, period
     FROM transcripts
     ORDER BY date
""", conn)
print(result)

=== Transcripts per market period ===
           period  count
0       Pre-COVID      6
1     COVID Crash      2
2  Rate Hike Peak      2
3         AI Boom      2
4    Tariff Shock      2

=== All Transcripts ===
   ticker  quarter        date          period
0    AAPL  Q1_2020  2020-01-28       Pre-COVID
1     JNJ  Q1_2020  2020-04-14       Pre-COVID
2     JPM  Q1_2020  2020-04-14       Pre-COVID
3    MSFT  Q1_2020  2020-04-29       Pre-COVID
4    AAPL  Q2_2020  2020-04-30     COVID Crash
5    AMZN  Q1_2020  2020-04-30       Pre-COVID
6     XOM  Q1_2020  2020-05-01       Pre-COVID
7     JPM  Q2_2020  2020-07-14     COVID Crash
8    MSFT  Q4_2022  2022-07-26  Rate Hike Peak
9    AAPL  Q4_2022  2022-10-27  Rate Hike Peak
10   MSFT  Q2_2023  2023-01-24         AI Boom
11   NVDA  Q2_2023  2023-08-23         AI Boom
12   AAPL  Q1_2025  2025-01-30    Tariff Shock
13   AMZN  Q1_2025  2025-05-01    Tariff Shock


In [13]:
# Always close the connection when done
conn.close()
print("Database connection closed.")

Database connection closed.
